# Edge IIoT - SMOTE

Column-level documentation and treatment decisions: [docs/preliminary.md](../../docs/preliminary.md)

In [8]:
LOAD_LARGE_CSV = True # Set to True if you want to load the DNN file instead of the ML one.

if LOAD_LARGE_CSV:
    filename = "DNN-EdgeIIoT-dataset"
else:
    filename = "ML-EdgeIIoT-dataset"

In [9]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from src.config import DATASETS

# Seleccionamos el dataset a analizar
dataset_name = "edge_iiot"
config = DATASETS[dataset_name]

print("\n--- Dataset Information ---")
print(f"Name: {dataset_name.upper()}")
print(f"Path: {config['processed_path']}\n")

csv_path = config['processed_path'] / f"{filename}.pkl"
print(f"CSV Path: {csv_path}\n")

print("Loading dataset... (This may take a while)")
df = pd.read_pickle(csv_path)

print(f"\nDataset loaded with shape: {df.shape}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

--- Dataset Information ---
Name: EDGE_IIOT
Path: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed

CSV Path: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed/DNN-EdgeIIoT-dataset.pkl

Loading dataset... (This may take a while)

Dataset loaded with shape: (2149035, 62)


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

# ----------

target_bin = 'Attack_label'
target_15c = 'Attack_type'
target_cols = [target_bin, target_15c]

y_bin = df[target_bin]
y_15c = df[target_15c]
X = df.drop(columns=target_cols)

# ----------
# Temporarily encode text columns to numbers for SMOTE

cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
if cat_cols:
    X[cat_cols] = encoder.fit_transform(X[cat_cols])

print(f"Text columns encoded: {cat_cols}")

# ----------

(
    X_train, X_test,
    y_train_bin,  y_test_bin,
    y_train_15,   y_test_15
) = train_test_split(
    X, y_bin, y_15c,
    test_size=0.2, random_state=42, stratify=y_15c
)

Text columns encoded: ['http.file_data', 'http.request.uri.query', 'http.referer', 'http.request.version', 'mqtt.msg', 'proto', 'http.request.path']


In [11]:
from collections import Counter

target_ratio = 0.15

# ----------

counts = Counter(y_train_15)
majority_class = max(counts, key=counts.get)
majority_count = counts[majority_class]
target_size = int(majority_count * target_ratio)

# ----------

strategy = {}
for class_label, count in counts.items():
    if class_label != majority_class:
        if count < target_size:
            strategy[class_label] = target_size

print("Strategy:", strategy)

Strategy: {'DDoS_ICMP': 193153, 'Password': 193153, 'SQL_injection': 193153, 'DDoS_HTTP': 193153, 'DDoS_UDP': 193153, 'XSS': 193153, 'Ransomware': 193153, 'Uploading': 193153, 'Port_Scanning': 193153, 'Vulnerability_scanner': 193153, 'Backdoor': 193153, 'DDoS_TCP': 193153, 'Fingerprinting': 193153, 'MITM': 193153}


In [12]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=strategy, random_state=42, k_neighbors=2)
X_train_res, y_train_15_res = smote.fit_resample(X_train, y_train_15)

# ----------
# Synchronize binary labels

normal_class_label = 'Normal'
y_train_bin_res = (y_train_15_res != normal_class_label).astype(int)

# ----------

print(f"Original Training Shape: {X_train.shape}")
print(f"Resampled Training Shape: {X_train_res.shape}")

Original Training Shape: (1719228, 60)
Resampled Training Shape: (3991835, 60)


In [13]:
# Reassemble train
df_train_res = pd.DataFrame(X_train_res, columns=X.columns)
df_train_res[target_bin] = y_train_bin_res
df_train_res[target_15c] = y_train_15_res

# Reassemble test
df_test = pd.DataFrame(X_test, columns=X.columns)
df_test[target_bin] = y_test_bin
df_test[target_15c] = y_test_15

# ----------
# Decode the categorical columns back to text

if cat_cols:
    df_train_res[cat_cols] = encoder.inverse_transform(df_train_res[cat_cols])
    df_test[cat_cols] = encoder.inverse_transform(df_test[cat_cols])

print(f"Training Dataset Shape (SMOTE): {df_train_res.shape}")
print(f"Testing Dataset Shape (Original): {df_test.shape}")

Training Dataset Shape (SMOTE): (3991835, 62)
Testing Dataset Shape (Original): (429807, 62)


In [14]:
train_output_path = config['processed_path'] / f"{filename}_train_SMOTE.pkl"
test_output_path = config['processed_path'] / f"{filename}_test.pkl"

df_train_res.to_pickle(train_output_path)
df_test.to_pickle(test_output_path)

print(f"\nSuccessfully saved training dataset to: {train_output_path}")
print(f"Successfully saved testing dataset to: {test_output_path}")


Successfully saved training dataset to: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed/DNN-EdgeIIoT-dataset_train_SMOTE.pkl
Successfully saved testing dataset to: /home/uo294319/ML-NIDS-IIoT/data/edge_iiot/processed/DNN-EdgeIIoT-dataset_test.pkl
